# DUSN-X: fine-tune LLM riêng bằng LoRA
Notebook này SFT LoRA cho Qwen2.5-0.5B-Instruct; không train từ đầu và không train router.
Dữ liệu đã review/attest/khóa tại commit `b95f5426fc1dcd68f95d491967081d2ae018bbda`. Test được hash để bảo đảm tính toàn vẹn nhưng trainer chỉ đọc train và validation; không dùng holdout v3.
Chọn **Runtime → Change runtime type → T4 GPU**. Chạy từng ô theo thứ tự, không dùng Run all. Sau mỗi chặng tải ZIP và SHA xuống máy.

In [ ]:
import os, sys, subprocess, pathlib, json
REPO_URL = "https://github.com/quocdoanhdang2-sketch/dusnx-platform.git"
REPO_REVISION = "b95f5426fc1dcd68f95d491967081d2ae018bbda"
assert len(REPO_REVISION) == 40
PROJECT = pathlib.Path("/content/dusnx-platform")
RUN = pathlib.Path("/content/dusnx-llm-run-b95f542")
DEPS = pathlib.Path("/content/dusnx-llm-deps-b95f542")
if not PROJECT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(PROJECT), "checkout", "--detach", REPO_REVISION], check=True)
os.chdir(PROJECT)
CODE_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert CODE_COMMIT == REPO_REVISION, f"Sai commit: {CODE_COMMIT}"
assert (3, 11) <= sys.version_info[:2] <= (3, 13), f"Python không được hỗ trợ: {sys.version}"
print("Repository commit:", CODE_COMMIT)
print("Kernel Python:", sys.version)
print("Project:", PROJECT)
print("Isolated dependency target:", DEPS)

# Colab's system Python may not include ensurepip. Install into an isolated
# target directory instead of creating a venv or changing Colab packages.
def call(*args):
    env = os.environ.copy()
    env["PYTHONPATH"] = os.pathsep.join([str(DEPS), str(PROJECT / "python/src")])
    command = [sys.executable, *map(str, args)]
    print("+", " ".join(command), flush=True)
    subprocess.run(command, cwd=PROJECT, env=env, check=True)

def call_capture(*args):
    env = os.environ.copy()
    env["PYTHONPATH"] = os.pathsep.join([str(DEPS), str(PROJECT / "python/src")])
    return subprocess.check_output([sys.executable, *map(str, args)], cwd=PROJECT, env=env, text=True).strip()

print("PASS: đã checkout đúng commit train và dữ liệu đã khóa")

## Cài đúng bộ thư viện mà không tạo venv
Ô này dùng Python có sẵn của Colab nhưng cài các phiên bản khóa trong `requirements/llm-sft.txt` vào thư mục riêng. Nó không thay các package của kernel Colab. Nếu cài lỗi, dừng và gửi toàn bộ lỗi pip hiện ra.

In [ ]:
DEPS.mkdir(parents=True, exist_ok=True)
call("-m", "pip", "install", "--disable-pip-version-check", "--target", str(DEPS), "--upgrade", "--ignore-installed", "-r", "requirements/llm-sft.txt")

gpu_check = call_capture("-c", "import torch; print(torch.__version__); print(int(torch.cuda.is_available())); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')")
print("PyTorch / CUDA / GPU:\n" + gpu_check)
gpu_lines = gpu_check.splitlines()
HAS_GPU = len(gpu_lines) >= 2 and gpu_lines[1].strip() == "1"
assert HAS_GPU, "GPU không hoạt động trong dependency target. Dừng, không train; kiểm tra Runtime GPU và gửi output ô này."
print("PASS: bộ thư viện đúng phiên bản, CUDA hoạt động")

## Audit dữ liệu đã khóa
Chạy ô này để kiểm tra manifest, attestation và độ dài token. Không chạy dự đoán trên test/holdout.

In [ ]:
call("python/scripts/prepare_llm_data.py", "--output", "/content/llm-audit", "--tokenizer-model", "Qwen/Qwen2.5-0.5B-Instruct", "--tokenizer-revision", "7ae557604adf67be50417f59c2c2f167def9a775", "--max-length", "512")
manifest = json.loads(pathlib.Path("datasets/llm_sft/manifest.json").read_text(encoding="utf-8"))
assert manifest.get("status") == "human_attested_locked_before_training", manifest.get("status")
assert pathlib.Path("datasets/llm_sft/human_test_attestation.json").is_file()
print("Manifest status:", manifest["status"])
print("Manifest SHA-256:", __import__("hashlib").sha256(pathlib.Path("datasets/llm_sft/manifest.json").read_bytes()).hexdigest())
print(pathlib.Path("/content/llm-audit/audit.json").read_text(encoding="utf-8"))
print("PASS: dữ liệu được attested và đã khóa")

## Train LoRA theo từng chặng
Cấu hình khóa: 3 epoch tối đa, batch 1, accumulation 8, context 512, LoRA r=8. Chạy epoch 1 rồi tải ZIP+SHA về máy trước khi tiếp tục. Nếu validation early stopping kết thúc training sớm, không ép chạy epoch tiếp.

In [ ]:
from google.colab import files
CONFIG = "configs/llm_sft.yaml"
RUN_FULL = True
assert HAS_GPU, "Full SFT cần GPU"
assert not RUN.exists() or not any(RUN.iterdir()), f"Output đã có dữ liệu: {RUN}. Không ghi đè; dùng phần restore hoặc tên run mới."
manifest = json.loads(pathlib.Path("datasets/llm_sft/manifest.json").read_text(encoding="utf-8"))
assert manifest.get("status") == "human_attested_locked_before_training"
assert pathlib.Path("datasets/llm_sft/human_test_attestation.json").is_file()
call("python/scripts/finetune_llm.py", "--config", CONFIG, "--output", RUN, "--stop-after-epoch", "1")
run_manifest = json.loads((RUN / "training_manifest.json").read_text(encoding="utf-8"))
print("Stage status:", run_manifest["status"])
print("Epoch:", run_manifest.get("epoch"))
print("PASS: chặng 1 kết thúc; tải backup trước khi tiếp tục")

In [ ]:
assert RUN.with_suffix(".zip").is_file(), "Thiếu ZIP checkpoint"
assert RUN.with_suffix(".zip.sha256").is_file(), "Thiếu SHA-256"
files.download(str(RUN.with_suffix(".zip")))
files.download(str(RUN.with_suffix(".zip.sha256")))
print("Đợi cả ZIP và SHA xuất hiện trong Downloads trước khi chạy chặng tiếp theo.")

### Chặng 2 và chặng 3
Sau khi xác nhận ZIP+SHA của chặng trước đã nằm trên máy, chạy ô sau với `NEXT_EPOCH = 2`. Tải ZIP+SHA. Sau khi xác nhận backup đó, sửa duy nhất `NEXT_EPOCH = 3`, chạy lại ô và tải backup cuối. Nếu manifest đã báo `training_complete`, không cần tiếp tục epoch.

In [ ]:
NEXT_EPOCH = 2
run_manifest_path = RUN / "training_manifest.json"
assert run_manifest_path.is_file(), "Không có run để resume; không chạy lại chặng 1 nếu đã bắt đầu."
run_manifest = json.loads(run_manifest_path.read_text(encoding="utf-8"))
if run_manifest.get("status") == "training_complete":
    print("Training đã hoàn tất theo early stopping; không chạy epoch tiếp.")
else:
    checkpoints = sorted(RUN.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
    assert checkpoints, "Không tìm thấy checkpoint để resume"
    last = checkpoints[-1]
    print("Resume from:", last, "to epoch:", NEXT_EPOCH)
    call("python/scripts/finetune_llm.py", "--config", CONFIG, "--output", RUN, "--resume", last, "--stop-after-epoch", str(NEXT_EPOCH))
    run_manifest = json.loads(run_manifest_path.read_text(encoding="utf-8"))
    print("Stage status:", run_manifest["status"], "epoch:", run_manifest.get("epoch"))
assert RUN.with_suffix(".zip").is_file()
assert RUN.with_suffix(".zip.sha256").is_file()
files.download(str(RUN.with_suffix(".zip")))
files.download(str(RUN.with_suffix(".zip.sha256")))

## Sau khi training hoàn tất
Không chạy merge GGUF hoặc đánh giá trong notebook này. Giữ ZIP+SHA của stage cuối; sau đó import candidate trên Windows và đánh giá paired qua Gateway theo `docs/WEEK3_LLM_TRAINING.md`. Không dùng test để chọn lại cấu hình và không mở holdout v3.